## Concatenate GO & Louvain Results

## Purpose: 

1. Take all the `GO` results produced from `./2_louvain_clustering_submission.py`. 
2. Subset to results where `p_fdr_benjamini_hochberg` <0.05
3. Save metadata about that `GO` result file
4. Concatenate all `Pandas DataFrames`. 
5. Output data as `gzip'd TSV` file. 
6. Remove all the `SLURM` and script intermediary files produced from `./2_louvain_clustering_submission.py` 

Separately, store the results from all `Louvain clusterings` in `dictionary` and then `pickle`.

In [1]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

import pandas as pd
pd.set_option('display.max_rows', 1000000)
pd.set_option('display.max_columns', 1000000)
pd.set_option('display.max_colwidth', 10000000)

import glob, os, shutil, pickle

from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm


## Load Data

In [2]:
parameter_combinations = [file.split("/")[-1].split(".")[0] for file in sorted(glob.glob("../../../../outputs/pickled_networks/differential_expression_tf_networks/*", recursive=True))]

parameter_combinations

['upstream_2500_score_900',
 'upstream_2500_score_950',
 'upstream_2500_score_990',
 'upstream_5000_score_900',
 'upstream_5000_score_950',
 'upstream_5000_score_990',
 'upstream_750_score_900',
 'upstream_750_score_950',
 'upstream_750_score_990']

## Concatenation Algorithm

In [3]:
# ONLY FOR USE IF DATA CONCATENATION IS LARGE

for parameter_combination in parameter_combinations:
    
    output_file = "/scratch/jve4pt/{}_output.txt".format(parameter_combination)
    error_file = "/scratch/jve4pt/{}_error.txt".format(parameter_combination)

    os.system("sbatch --output='{}' --account=platiglab --error='{}' --partition=standard --mem=20GB -N1 -n1 --wrap='python3 ./3_concatenate_large_GO_data.py {}'".format(output_file, error_file, parameter_combination))

Submitted batch job 1691186


0

Submitted batch job 1691187


0

Submitted batch job 1691188


...

0

Submitted batch job 1691193


0

Submitted batch job 1691194


0

In [5]:
# ONLY FOR /scratch/jve4pt/

# list with all the Gene Ontology analysis results in one
meta_GO_df = []

"""
for GO results for all possible parameter combinations, iterations, and clusters, 
retrieve them via `glob`, subset to FDR p < 0.05, 
save info about parameter combination, iteration and cluster
and finally concatenate all these together

# OPTIONAL: filtering GO terms by keywords

"""



for parameter_combination in parameter_combinations:
    
    parameter_combination
    
                
    tmp_files = glob.glob(
        "/scratch/jve4pt/*{}*.tsv.gz".format(parameter_combination)
    )

    if len(tmp_files)==0: 
        print("{} {}".format(parameter_combination))

    for file in tmp_files: 

        df = pd.read_csv(file , sep="\t")

        df = df[(df["p_uncorrected"]<0.05) & (df["NS"] =="BP")]

#             # subset to cardiac relevant tissue 
#             df =df[(df["name"].str.contains("blood")) | (df["name"].str.contains("heart")) | (df["name"].str.contains("vascul")) | (df["name"].str.contains("circul")) | (df["name"].str.contains("cardiac"))]

        if df.index.size>0:


            meta_GO_df.append(df)


meta_GO_df = pd.concat(meta_GO_df)

'\nfor GO results for all possible parameter combinations, iterations, and clusters, \nretrieve them via `glob`, subset to FDR p < 0.05, \nsave info about parameter combination, iteration and cluster\nand finally concatenate all these together\n\n# OPTIONAL: filtering GO terms by keywords\n\n'

'upstream_2500_score_900'

'upstream_2500_score_950'

'upstream_2500_score_990'

'upstream_5000_score_900'

'upstream_5000_score_950'

'upstream_5000_score_990'

'upstream_750_score_900'

'upstream_750_score_950'

'upstream_750_score_990'

In [ ]:
# # dataframe with all the Gene Ontology analysis results in one
# meta_GO_df = pd.DataFrame()

# """
# for GO results for all possible parameter combinations, iterations, and clusters, 
# retrieve them via `glob`, subset to FDR p < 0.05, 
# save info about parameter combination, iteration and cluster
# and finally concatenate all these together

# # OPTIONAL: filtering GO terms by keywords

# """



# for parameter_combination in parameter_combinations:
    
#     parameter_combination
    
#     for iteration in range(1000): 
                
#         tmp_files = glob.glob(
#             "/scratch/jve4pt/gene_ontology/differential_expression_networks/{}-iter_{}/*.tsv".format(parameter_combination, iteration)
#         )
        
#         if len(tmp_files)==0: 
#             print("{} {}".format(parameter_combination, iteration))
            
#         for file in tmp_files: 
            
#             cluster = file.split("-")[-1].split(".")[0]
                        
#             df = pd.read_csv(file , sep="\t")
            
#             df = df[df["p_uncorrected"]<0.05].copy()
                        
# #             # subset to cardiac relevant tissue 
# #             df =df[(df["name"].str.contains("blood")) | (df["name"].str.contains("heart")) | (df["name"].str.contains("vascul")) | (df["name"].str.contains("circul")) | (df["name"].str.contains("cardiac"))]
            
#             if df.index.size>0:
                
#                 df["Parameter_Combination"] = parameter_combination
#                 df["Iteration"] = iteration 
#                 df["Cluster"] = cluster
            
#                 meta_GO_df = pd.concat([meta_GO_df, df])



In [6]:
meta_GO_df.index.size

meta_GO_df.head()

18114924

,# GO,NS,enrichment,name,ratio_in_study,ratio_in_pop,p_uncorrected,depth,study_count,p_fdr_bh,study_items,Parameter_Combination,Iteration,Cluster
0,GO:0043588,BP,e,skin development,5/73,58/21965,0.000001,4,5,0.017343,"col3a1, col5a1, dhcr24, gnas, nfkbiz",upstream_2500_score_900,0,cluster_3
1,GO:0030154,BP,e,cell differentiation,13/73,967/21965,0.000016,3,13,0.098247,"ccdc63, etv4, mdk, nell1, nr2f1, nr2f6, nrg1, sfrp4, slit3, smad6, tdrd7, tenm4, tnfrsf12a",upstream_2500_score_900,0,cluster_3
2,GO:0031589,BP,e,cell-substrate adhesion,3/73,22/21965,0.000052,3,3,0.212585,"coro1a, smad6, vwf",upstream_2500_score_900,0,cluster_3
3,GO:0003180,BP,e,aortic valve morphogenesis,3/73,28/21965,0.000109,4,3,0.334273,"axin2, slit3, smad6",upstream_2500_score_900,0,cluster_3
4,GO:0042158,BP,e,lipoprotein biosynthetic process,2/73,6/21965,0.000162,6,2,0.334273,"abca1, apob",upstream_2500_score_900,0,cluster_3


## Output `DataFrame` in `gzip` Format 

In [7]:
meta_GO_df.to_csv(
    "../../../../outputs/gene_ontology_analysis/differential_expression_networks/all_GO_analyses_p_uncorrected_0.05.tsv.gz", 
    sep="\t", 
    compression='gzip', 
    index=False
)

Make sure that file is properly formatted

In [8]:
tmp = pd.read_csv(
    "../../../../outputs/gene_ontology_analysis/differential_expression_networks/all_GO_analyses_p_uncorrected_0.05.tsv.gz", 
    sep="\t",
    compression='gzip',
)

tmp.index.size
tmp.head()

18114924

,# GO,NS,enrichment,name,ratio_in_study,ratio_in_pop,p_uncorrected,depth,study_count,p_fdr_bh,study_items,Parameter_Combination,Iteration,Cluster
0,GO:0043588,BP,e,skin development,5/73,58/21965,0.000001,4,5,0.017343,"col3a1, col5a1, dhcr24, gnas, nfkbiz",upstream_2500_score_900,0,cluster_3
1,GO:0030154,BP,e,cell differentiation,13/73,967/21965,0.000016,3,13,0.098247,"ccdc63, etv4, mdk, nell1, nr2f1, nr2f6, nrg1, sfrp4, slit3, smad6, tdrd7, tenm4, tnfrsf12a",upstream_2500_score_900,0,cluster_3
2,GO:0031589,BP,e,cell-substrate adhesion,3/73,22/21965,0.000052,3,3,0.212585,"coro1a, smad6, vwf",upstream_2500_score_900,0,cluster_3
3,GO:0003180,BP,e,aortic valve morphogenesis,3/73,28/21965,0.000109,4,3,0.334273,"axin2, slit3, smad6",upstream_2500_score_900,0,cluster_3
4,GO:0042158,BP,e,lipoprotein biosynthetic process,2/73,6/21965,0.000162,6,2,0.334273,"abca1, apob",upstream_2500_score_900,0,cluster_3


## Remove Original Output from Script + `SLURM`

In [ ]:
# if os.path.exists("../../../../outputs/gene_ontology_analysis/differential_expression_networks/output/"): 
#     shutil.rmtree("../../../../outputs/gene_ontology_analysis/differential_expression_networks/output/")
    
# if os.path.exists("../../../../outputs/gene_ontology_analysis/differential_expression_networks/SLURM/"):    
#     shutil.rmtree("../../../../outputs/gene_ontology_analysis/differential_expression_networks/SLURM/")

## Create Dict of Louvain Results and Pickle

In [9]:
louvain_results = {}

# load all the pickled louvain community clustering results 
# into a dict such that it's organized by parameter combination and iteration

for parameter_combination in parameter_combinations:
        
    louvain_results[parameter_combination]={}
    
    for iteration in range(1000): 
        louvain_results[parameter_combination][iteration] = pickle.load(open("/scratch/jve4pt/louvain_clustering/differential_expression_networks/{}/{}-iter_{}.pkl".format(parameter_combination, parameter_combination, iteration), 'rb'))


In [10]:
pickle.dump(louvain_results, open("../../../../outputs/louvain_clustering/all_louvain_clusterings.pkl", 'wb'))

In [11]:
tmp_test = pickle.load(open("../../../../outputs/louvain_clustering/all_louvain_clusterings.pkl", 'rb'))

In [12]:
tmp_test["upstream_2500_score_900"][1][0]

{'abcc5',
 'adam33',
 'adgrg3',
 'aldh1l1',
 'apob',
 'astn2',
 'atcay',
 'atf3',
 'atp1a3',
 'axin2',
 'b4galt6',
 'bhmt',
 'car3',
 'ccdc63',
 'col5a1',
 'coro1a',
 'cpn1',
 'creb3l1',
 'ctcfl',
 'dhcr7',
 'dop1b',
 'dusp8',
 'fads2',
 'fbxo32',
 'fgfr2',
 'galnt10',
 'gramd1b',
 'h2-q6',
 'hipk2',
 'hnf1b',
 'hspb8',
 'il1r1',
 'kcnj6',
 'kif12',
 'lhx3',
 'lims2',
 'lipo3',
 'lpl',
 'lratd1',
 'map1b',
 'mbnl2',
 'mttp',
 'mvb12b',
 'mvk',
 'myh11',
 'nacc2',
 'nfkbiz',
 'nptx2',
 'nr5a2',
 'nr6a1',
 'nrg1',
 'onecut2',
 'pbx3',
 'pdgfb',
 'pip4k2b',
 'plcd3',
 'psca',
 'rims2',
 'robo2',
 'rora',
 'rreb1',
 'rsad2',
 'ryr2',
 'sct',
 'synpo',
 'timp3',
 'tnc',
 'usp13',
 'vwf',
 'xdh'}

## Remove Extraneous Clustering Data Folders

In [ ]:
# folders = glob.glob("../../../../outputs/louvain_clustering/upstream_*")
# folders

In [ ]:
# for folder in folders: 
#     shutil.rmtree(folder)

## Remove this Later

In [13]:
def process_combination_iteration(parameter_combination, iteration):
    results = []
    tmp_files = glob.glob(
        "/scratch/jve4pt/gene_ontology/differential_expression_networks/{}-iter_{}/*.tsv".format(parameter_combination, iteration)
    )
    
    assert len(tmp_files) > 0
    
    for file in tmp_files:
        cluster = file.split("-")[-1].split(".")[0]
        df = pd.read_csv(file, sep="\t")
        df = df[(df["p_uncorrected"] < 0.05) & (df["NS"] == "BP")]
        
        if df.index.size > 0:
            df["Parameter_Combination"] = parameter_combination
            df["Iteration"] = iteration
            df["Cluster"] = cluster
            results.append(df)
            
    return results

all_results = []
with ThreadPoolExecutor() as executor:
    futures = [executor.submit(process_combination_iteration, parameter_combination, iteration) for parameter_combination in parameter_combinations for iteration in range(1000)]
    for future in tqdm(as_completed(futures), total=len(parameter_combinations) * 1000):
        all_results.extend(future.result())

# Combine all results into a single DataFrame if needed
if all_results:
    final_df = pd.concat(all_results, ignore_index=True)

100%|██████████| 9000/9000 [27:10<00:00,  5.52it/s]  


In [ ]:
# baseline_append = pd.DataFrame()

# for parameter_combination in parameter_combinations:
    
#     parameter_combination
    
#     for iteration in range(1000): 
                
#         tmp_files = glob.glob(
#             "/scratch/jve4pt/gene_ontology/differential_expression_networks/{}-iter_{}/*.tsv".format(parameter_combination, iteration), 
#         )
        
#         assert len(tmp_files)>0
            
#         for file in tmp_files: 
            
#             cluster = file.split("-")[-1].split(".")[0]
                        
#             df = pd.read_csv(file , sep="\t")
            
#             df = df[(df["p_uncorrected"]<0.05) & (df["NS"] == "BP")]
                        
# #             # subset to cardiac relevant tissue 
# #             df =df[(df["name"].str.contains("blood")) | (df["name"].str.contains("heart")) | (df["name"].str.contains("vascul")) | (df["name"].str.contains("circul")) | (df["name"].str.contains("cardiac"))]
            
#             if df.index.size>0:
                
#                 df["Parameter_Combination"] = parameter_combination
#                 df["Iteration"] = iteration 
#                 df["Cluster"] = cluster
            
#                 baseline_append = pd.concat([baseline_append, df])

In [14]:
final_df.index.size
final_df.head()

18114924

,# GO,NS,enrichment,name,ratio_in_study,ratio_in_pop,p_uncorrected,depth,study_count,p_fdr_bh,study_items,Parameter_Combination,Iteration,Cluster
0,GO:0042157,BP,e,lipoprotein metabolic process,4/85,18/21965,6.131241e-07,5,4,0.007544,"apoa1, apom, ldlr, pcsk9",upstream_2500_score_900,23,cluster_1
1,GO:0060716,BP,e,labyrinthine layer blood vessel development,4/85,27/21965,3.424300e-06,5,4,0.021068,"ccn1, nsdhl, plcd3, wnt2",upstream_2500_score_900,23,cluster_1
2,GO:0016055,BP,e,Wnt signaling pathway,7/85,218/21965,2.206345e-05,6,7,0.090497,"frat1, fzd7, hhex, hic1, sfrp4, wnt2, wnt4",upstream_2500_score_900,23,cluster_1
3,GO:0060070,BP,e,canonical Wnt signaling pathway,5/85,102/21965,4.780178e-05,7,5,0.147050,"frat1, fzd7, sfrp4, wnt2, wnt4",upstream_2500_score_900,23,cluster_1
4,GO:0090263,BP,e,positive regulation of canonical Wnt signaling pathway,5/85,107/21965,6.009553e-05,8,5,0.147895,"adnp, frat1, hhex, sfrp4, sox4",upstream_2500_score_900,23,cluster_1


In [15]:
final_df.to_csv(
    "../../../../outputs/gene_ontology_analysis/differential_expression_networks/baseline_networks_GO_p_uncorrected_0.05.tsv.gz", 
    compression="gzip", 
    index=False, 
    sep="\t"
)

In [16]:
tmp = pd.DataFrame(meta_GO_df[meta_GO_df["p_fdr_bh"]<0.05].groupby(["name"]).median(numeric_only=True))
tmp.head()

,p_uncorrected,depth,study_count,p_fdr_bh,Iteration
name,,,,,
BMP signaling pathway,0.000005,9.0,4.0,0.035474,422.0
NMDA glutamate receptor clustering,0.000070,8.0,2.0,0.045680,553.0
Notch signaling pathway,0.000040,6.0,5.0,0.044323,254.0
Roundabout signaling pathway,0.000037,6.0,2.0,0.041712,570.0
SMAD protein signal transduction,0.000010,7.0,4.0,0.028864,509.0


In [17]:
tmp = pd.DataFrame(final_df.groupby(["name"])["p_fdr_bh"].median(numeric_only=True))

tmp.head()

,p_fdr_bh
name,
'de novo' L-methionine biosynthetic process,0.557422
"1-phosphatidyl-1D-myo-inositol 4,5-bisphosphate biosynthetic process",0.769231
10-formyltetrahydrofolate catabolic process,0.655602
3'-UTR-mediated mRNA stabilization,1.000000
9-cis-retinoic acid biosynthetic process,1.000000


In [19]:
meta_GO_df = meta_GO_df[meta_GO_df["p_fdr_bh"]<0.05]

meta_GO_df = pd.DataFrame(meta_GO_df["name"].value_counts())

In [20]:
meta_GO_df.head(n=100)

,count
name,
cholesterol metabolic process,9134
positive regulation of transcription by RNA polymerase II,6059
cholesterol homeostasis,5103
steroid metabolic process,5058
multicellular organism development,4380
lipoprotein metabolic process,3860
sterol biosynthetic process,3731
positive regulation of DNA-templated transcription,3494
positive regulation of TRAIL-activated apoptotic signaling pathway,3252
